# Shared-or-solo model

This notebook runs the model from `source/shared_model.py` step by step.

### Goal
Predict whether a purchase is **shared** (a couple expense, "Group") or **solo**.

### Input
One row per labeled purchase: merchant, amount, category, detailed category, payment channel, and the label `Solo / Group`. The spreadsheet is read from `data/Transactions.xlsx` or `data/raw/Transactions.xlsx` (sheet `data_pd`).

### How it works
1. **Load, clean, split.** The purchases are split two ways:
   - **Split A (random):** most test stores also appear in training.
   - **Split B (new merchants):** no test store appears in training. This is the harder case: will the model work for a store it has never seen?
2. **Merchant embeddings.** Each merchant name is turned into numbers with a small pretrained language model, so that, for example, "Trader Joe's" ends up close to "grocery store" even if it never appeared in training.
3. **Models.** Two ridge logistic regressions are trained and compared:
   - **without embeddings:** merchant, categories, channel and amount
   - **with embeddings:** the same features plus the merchant-name embeddings

### Reading the results
- **ROC-AUC:** 0.5 is random guessing, 1.0 is perfect.
- **PR-AUC:** how well the model finds shared purchases. Random guessing scores the share of shared purchases, so compare against that number.
- **change:** embeddings minus no embeddings. A positive number means the embeddings helped.

Run the cells from top to bottom (**Run All**). The embedding model downloads once on first use, and the 5-fold comparison takes a few minutes.

> The outputs include summary numbers only, but clear outputs before committing this notebook anyway.


## Setup

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

from sentence_transformers import SentenceTransformer
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GridSearchCV, GroupKFold, StratifiedGroupKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 0
SHEET = "data_pd"

# Find the repo root (the folder that contains source/), whether the notebook runs from
# the repo root or from notebooks/.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "source" / "shared_model.py").exists())

# The script expects data/Transactions.xlsx; also accept data/raw/.
CANDIDATES = [ROOT / "data" / "Transactions.xlsx", ROOT / "data" / "raw" / "Transactions.xlsx"]
PATH = next((p for p in CANDIDATES if p.exists()), None)
assert PATH is not None, f"Spreadsheet not found. Looked in: {[str(p) for p in CANDIDATES]}"
print("Reading", PATH.relative_to(ROOT))

Reading data\raw\Transactions.xlsx


## Part 1: load, clean, split

In [2]:
# 1. Load and rename columns
df = pd.read_excel(PATH, sheet_name=SHEET).rename(columns={
    "Transaction ID": "tid",
    "Merchant": "merchant",
    "Amount (USD)": "amount",
    "Detailed category": "det",
    "Payment channel": "channel",
    "Solo / Group": "label",
})

# 2. Keep labeled purchases only; drop refunds (negative amounts).
#    Repeated identical purchases (transit fares, subscriptions) are real, so no deduping.
df = df[df["label"].isin(["Solo", "Group"]) & (df["amount"] > 0)].copy()

# 3. Clean text fields and rebuild the broad category from the detailed one
PRIMARIES = [
    "FOOD_AND_DRINK", "GENERAL_MERCHANDISE", "GENERAL_SERVICES", "HOME_IMPROVEMENT",
    "RENT_AND_UTILITIES", "TRANSPORTATION", "TRAVEL", "ENTERTAINMENT", "MEDICAL",
    "PERSONAL_CARE", "GOVERNMENT_AND_NON_PROFIT", "LOAN_PAYMENTS", "BANK_FEES",
    "TRANSFER_IN", "TRANSFER_OUT", "INCOME",
]
df["merchant"] = df["merchant"].astype(str).str.strip()
df["det"] = df["det"].fillna("UNKNOWN")
df["channel"] = df["channel"].fillna("unknown")
df["cat"] = df["det"].map(lambda d: next((p for p in PRIMARIES if d.startswith(p)), "OTHER"))

# 4. Target and amount feature
df["y"] = (df["label"] == "Group").astype(int)
df["log_amount"] = np.log1p(df["amount"])

print(f"Labeled purchases: {len(df)}, shared: {df['y'].mean():.1%}, merchants: {df['merchant'].nunique()}")

Labeled purchases: 1980, shared: 21.3%, merchants: 537


Share of shared (group) purchases by broad category:

In [3]:
by_cat = (df.groupby("cat")["y"].agg(purchases="size", shared_share="mean")
            .sort_values("purchases", ascending=False))
display(by_cat.style.format({"shared_share": "{:.0%}"}))

,purchases,shared_share
cat,,
FOOD_AND_DRINK,553,18%
TRANSPORTATION,354,4%
GENERAL_MERCHANDISE,248,31%
RENT_AND_UTILITIES,170,58%
ENTERTAINMENT,151,15%
PERSONAL_CARE,143,9%
GENERAL_SERVICES,127,8%
TRAVEL,70,23%
MEDICAL,68,25%


In [4]:
# 5a. Split A: random 80/20, same share of shared purchases on both sides
trainA, testA = train_test_split(df, test_size=0.2, stratify=df["y"], random_state=SEED)

# 5b. Split B: merchant-held-out 80/20, also balanced on the shared share
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
train_idx, test_idx = next(sgkf.split(df, df["y"], groups=df["merchant"]))
trainB, testB = df.iloc[train_idx], df.iloc[test_idx]
assert not set(testB["merchant"]) & set(trainB["merchant"]), "Merchant leaked across split B"


def describe(train, test):
    return {
        "train rows": len(train),
        "test rows": len(test),
        "shared in test": f"{test['y'].mean():.1%}",
        "test merchants seen in training": f"{test['merchant'].isin(train['merchant']).mean():.0%}",
    }


display(pd.DataFrame({
    "Split A (random)": describe(trainA, testA),
    "Split B (merchant-held-out)": describe(trainB, testB),
}))

,Split A (random),Split B (merchant-held-out)
train rows,1584,1584
test rows,396,396
shared in test,21.2%,21.2%
test merchants seen in training,85%,0%


## Part 2: merchant embeddings

In [5]:
MODEL_NAME = "all-MiniLM-L6-v2"           # small pretrained encoder, 384 numbers per name
PCA_SIZES = [4, 8, 16, 32]                # hyperparameter: embedding dimensions kept
C_VALUES = np.logspace(-3, 2, 10)         # hyperparameter: inverse penalty strength

STORE_TYPES = {
    "st_grocery": "grocery store or supermarket",
    "st_restaurant": "restaurant",
    "st_coffee": "coffee shop or cafe",
    "st_fastfood": "fast food",
    "st_delivery": "food delivery app",
    "st_bar": "bar or nightlife",
    "st_home": "home goods or furniture store",
    "st_hardware": "hardware store",
    "st_utilities": "electricity, gas or internet provider",
    "st_phone": "mobile phone carrier",
    "st_rent": "rent or housing payment",
    "st_pharmacy": "pharmacy or drugstore",
    "st_clothing": "clothing store",
    "st_beauty": "beauty or personal care",
    "st_gym": "gym or fitness class",
    "st_rideshare": "rideshare or taxi",
    "st_transit": "public transit fare",
    "st_travel": "airline, hotel or travel booking",
    "st_streaming": "streaming or entertainment subscription",
    "st_marketplace": "online marketplace",
}
STORE_COLS = list(STORE_TYPES)
CAT_COLS = ["merchant", "cat", "det", "channel"]

# Embed every merchant name once (normalized, so a dot product is cosine similarity).
# Nothing here is learned from the labels, so it is safe to compute for all rows up front.
encoder = SentenceTransformer(MODEL_NAME)
merchants = sorted(df["merchant"].unique())
merchant_vecs = encoder.encode(merchants, normalize_embeddings=True, show_progress_bar=True)
type_vecs = encoder.encode(list(STORE_TYPES.values()), normalize_embeddings=True)
EMB_COLS = [f"emb_{i}" for i in range(merchant_vecs.shape[1])]

emb_table = pd.DataFrame(merchant_vecs, columns=EMB_COLS)
emb_table[STORE_COLS] = merchant_vecs @ type_vecs.T      # similarity to each store type
emb_table["merchant"] = merchants
print(f"Embedded {len(merchants)} merchants into {merchant_vecs.shape[1]} dimensions")

Embedded 537 merchants into 384 dimensions


## Part 3: models and evaluation

In [6]:
def with_embeddings(part):
    """Attach each row's merchant embedding and store-type scores."""
    return part.merge(emb_table, on="merchant", how="left")


def make_search(use_embeddings, train):
    """Ridge logistic regression, optionally with embeddings. PCA sits inside the pipeline,
    so during cross-validation it is refit on each training fold."""
    steps = [("cats", OneHotEncoder(handle_unknown="ignore"), CAT_COLS)]
    grid = {"clf__C": C_VALUES}
    if use_embeddings:
        steps += [
            ("num", StandardScaler(), ["log_amount"] + STORE_COLS),
            ("emb", Pipeline([("pca", PCA(random_state=SEED)), ("scale", StandardScaler())]), EMB_COLS),
        ]
        grid["pre__emb__pca__n_components"] = PCA_SIZES
    else:
        steps += [("num", StandardScaler(), ["log_amount"])]

    pipe = Pipeline([
        ("pre", ColumnTransformer(steps)),
        ("clf", LogisticRegression(class_weight="balanced", max_iter=5000)),
    ])
    # Folds grouped by merchant: hyperparameters are tuned for stores the model has not seen.
    folds = list(GroupKFold(n_splits=5).split(train, train["y"], groups=train["merchant"]))
    return GridSearchCV(pipe, grid, scoring="average_precision", cv=folds, n_jobs=-1)


def evaluate(train, test):
    """Tune both models on `train`, score them once on `test`."""
    tr, te = with_embeddings(train), with_embeddings(test)
    out = {}
    for use_emb in (False, True):
        search = make_search(use_emb, tr).fit(tr, tr["y"])
        p = search.predict_proba(te)[:, 1]
        out[use_emb] = {
            "roc": roc_auc_score(te["y"], p),
            "pr": average_precision_score(te["y"], p),
            "params": search.best_params_,
        }
    return out

Split A uses one random test set. Split B uses all five merchant-held-out folds, because one fold alone can be lucky. This cell takes a few minutes.

In [7]:
runs = {"A (random)": [evaluate(trainA, testA)], "B (new merchants, 5 folds)": []}
for k, (tr_idx, te_idx) in enumerate(sgkf.split(df, df["y"], groups=df["merchant"])):
    print(f"Split B, fold {k + 1} of 5...")
    runs["B (new merchants, 5 folds)"].append(evaluate(df.iloc[tr_idx], df.iloc[te_idx]))
print("Done.")

Split B, fold 1 of 5...
Split B, fold 2 of 5...
Split B, fold 3 of 5...
Split B, fold 4 of 5...
Split B, fold 5 of 5...
Done.


## Results

In [8]:
def summarize(values):
    values = np.array(values)
    return f"{values.mean():.3f}" if len(values) == 1 else f"{values.mean():.3f} \u00b1 {values.std():.3f}"


rows = []
for split_name, split_runs in runs.items():
    row = {"split": split_name}
    for metric, label in [("roc", "ROC-AUC"), ("pr", "PR-AUC")]:
        base = [r[False][metric] for r in split_runs]
        emb = [r[True][metric] for r in split_runs]
        row[f"{label} no embeddings"] = summarize(base)
        row[f"{label} + embeddings"] = summarize(emb)
        row[f"{label} change"] = f"{np.mean(emb) - np.mean(base):+.3f}"
    rows.append(row)

summary = pd.DataFrame(rows).set_index("split").T
print(f"Random guessing: ROC-AUC 0.500, PR-AUC {df['y'].mean():.3f} (share of shared purchases)")
print("Split B shows the average \u00b1 spread across 5 folds.")
display(summary)

Random guessing: ROC-AUC 0.500, PR-AUC 0.213 (share of shared purchases)
Split B shows the average ± spread across 5 folds.


split,A (random),"B (new merchants, 5 folds)"
ROC-AUC no embeddings,0.926,0.876 ± 0.050
ROC-AUC + embeddings,0.924,0.881 ± 0.047
ROC-AUC change,-0.002,+0.004
PR-AUC no embeddings,0.806,0.736 ± 0.065
PR-AUC + embeddings,0.799,0.726 ± 0.080
PR-AUC change,-0.007,-0.010


Every individual test set, with the hyperparameters cross-validation picked:

In [9]:
detail = []
for split_name, split_runs in runs.items():
    for i, r in enumerate(split_runs, start=1):
        detail.append({
            "split": split_name,
            "fold": i,
            "ROC-AUC no emb": r[False]["roc"],
            "ROC-AUC + emb": r[True]["roc"],
            "PR-AUC no emb": r[False]["pr"],
            "PR-AUC + emb": r[True]["pr"],
            "C no emb": r[False]["params"]["clf__C"],
            "C + emb": r[True]["params"]["clf__C"],
            "PCA size": r[True]["params"]["pre__emb__pca__n_components"],
        })
display(pd.DataFrame(detail).set_index(["split", "fold"]).round(3))

ROC-AUC no emb  ROC-AUC + emb  PR-AUC no emb  \
split                      fold                                                 
A (random)                 1              0.926          0.924          0.806   
B (new merchants, 5 folds) 1              0.942          0.947          0.827   
                           2              0.883          0.866          0.720   
                           3              0.804          0.806          0.643   
                           4              0.837          0.877          0.701   
                           5              0.915          0.907          0.790   

                                 PR-AUC + emb  C no emb  C + emb  PCA size  
split                      fold                                             
A (random)                 1            0.799     7.743    7.743         4  
B (new merchants, 5 folds) 1            0.844     2.154    7.743         4  
                           2            0.636   100.000    7.743         4  
                           3            0.643    27.826    7.743        16  
                           4            0.729     0.599    7.743         4  
                           5            0.778     7.743   27.826         4